# 02 SparkSession

## Goal
Understand what a SparkSession is, what it gives you, and how it works in Databricks.

## Why it matters
SparkSession is the first thing every Spark program creates, and "what is SparkSession?" is a standard interview question. Knowing what hangs off it makes every later lesson easier.

## Concept
**SparkSession** is the single entry point to Spark functionality. Through it you can:

- Create DataFrames (`spark.createDataFrame`)
- Read data (`spark.read`)
- Run SQL (`spark.sql`)
- Read and set configuration (`spark.conf`)
- Access the catalog of tables and databases (`spark.catalog`)

In Databricks, a SparkSession is already created and available as `spark`. In your own scripts outside Databricks, you create it yourself.

## Real-world example
A nightly ETL job starts, creates one SparkSession, reads raw files with `spark.read`, transforms them, and writes tables. Everything in the job goes through that one object.

## SparkSession vs SparkContext

| Name | Role |
|---|---|
| SparkContext | The older entry point (Spark 1.x). Connects to the cluster and creates RDDs |
| SQLContext / HiveContext | Older entry points for DataFrames and SQL |
| SparkSession | Unified entry point since Spark 2.0. Wraps all of the above |

SparkSession replaced SQLContext and HiveContext, and it holds a SparkContext inside it.

## Creating a SparkSession outside Databricks

```python
# Import the SparkSession class
from pyspark.sql import SparkSession

# Build a session with an application name.
# getOrCreate() reuses a running session or creates a new one.
spark = (
    SparkSession.builder
    .appName("MyApp")
    .getOrCreate()
)
```

In Databricks, `getOrCreate()` simply returns the session that already exists, so it is safe to run.

## 1. Inspect the session

**What it does:** prints the `spark` object, the Spark version and the object type.
**Why it matters:** confirms the session exists and tells you which Spark version you are using.
**Expected output:** a SparkSession description, a version string, and a type that mentions `SparkSession`.

In [0]:
# 'spark' is the SparkSession that Databricks creates for us automatically.
# Printing it shows that the session object exists.
print(spark)

# spark.version tells us which Apache Spark version is running.
print("Spark version:", spark.version)

# type() shows what kind of object 'spark' is (it should mention SparkSession).
print("Type:", type(spark))

## 2. getOrCreate in Databricks

**What it does:** asks for a session using the builder and compares it to `spark`.
**Why it matters:** shows that `getOrCreate()` reuses the existing session instead of making a new one.
**Expected output:** `True` or `False`. On serverless the result can differ from classic compute, so write down what you see.

In [0]:
# Import the SparkSession class so we can use its builder.
from pyspark.sql import SparkSession

# getOrCreate() returns the session that is already running.
# If none existed, it would create a new one.
same_spark = SparkSession.builder.getOrCreate()

# Check whether we got back the very same object as 'spark'.
print("Same session as spark?", same_spark is spark)

## 3. What you can do with SparkSession

**What it does:** creates a DataFrame from a Python list, generates numbers with `range`, and runs a SQL query.
**Why it matters:** these three calls (`createDataFrame`, `range`, `sql`) are the most common ways to start working with data.
**Expected output:** a 4-row table with salaries, a 5-row table of ids 0 to 4, and a one-row table with `one` and `greeting`.

In [0]:
# 1. Create a DataFrame from a Python list.
#    Each tuple is one row of data.
data = [
    (1, "Amal", "India", 21000),
    (2, "Senthil", "USA", 150000),
    (3, "John", "UK", 18000),
    (4, "Vivek", "UAE", 25000),
]

# The column names, in the same order as the values in each tuple.
columns = ["id", "name", "country", "salary"]

# createDataFrame() builds a distributed table from the data and column names.
df = spark.createDataFrame(data, columns)

# show() prints the DataFrame as a plain text table.
df.show()

# 2. spark.range(5) generates a DataFrame with one column 'id' holding 0 to 4.
#    It is handy for quick tests when you have no data.
spark.range(5).show()

# 3. spark.sql() runs a SQL query and returns the result as a DataFrame.
#    Here we select two literal values, so no table is needed.
spark.sql("SELECT 1 AS one, 'hello' AS greeting").show()

## 4. Catalog and configuration

**What it does:** reads the current catalog and schema, then tries to read one Spark setting.
**Why it matters:** the catalog tells you where tables are created by default. Configs control how Spark runs, but serverless restricts many of them.
**Expected output:** your current catalog and database (often `workspace` and `default`), and a shuffle partitions value or a "not readable" message.

In [0]:
# The catalog is the list of data objects Spark can see (catalogs, schemas, tables).
# try/except keeps the notebook running if a call isn't supported on this compute.
try:
    # The catalog we are currently working in.
    print("Current catalog:", spark.catalog.currentCatalog())
    # The schema (database) we are currently working in.
    print("Current database:", spark.catalog.currentDatabase())
except Exception as e:
    print("Not available here:", e)

# spark.conf reads Spark settings.
# 'shuffle partitions' controls how many pieces data is split into after a shuffle.
# Serverless may not let us read it, so we handle that case.
try:
    print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception as e:
    print("Config not readable here:", e)

## 5. SparkContext on serverless

**What it does:** tries to read the application name through `spark.sparkContext`.
**Why it matters:** SparkContext is the older entry point inside SparkSession. On serverless it is restricted, which is an important difference from classic compute.
**Expected output:** most likely the "not available" message. Write down what you see.

In [0]:
# SparkContext is the older, lower-level entry point that SparkSession wraps.
# On classic compute we can reach it with spark.sparkContext.
# On serverless this is restricted, so we expect an error and we catch it.
try:
    print("App name:", spark.sparkContext.appName)
except Exception as e:
    # type(e).__name__ prints just the error type, keeping the output short.
    print("sparkContext is not available on serverless:", type(e).__name__)

## What Spark is doing internally

```
Your Python code (driver)
        |
        v
   SparkSession   <-- entry point
        |
        v
  Builds query plans
        |
        v
  Cluster runs the work on executors
        |
        v
  Results return to the driver
```

The SparkSession lives in the **driver**. It builds the plan, and the cluster executes it. One session per application is the normal pattern.

## Troubleshooting log

| Problem | Fix |
|---|---|
| `sparkContext` raises an error on serverless | Expected, serverless restricts it. Use `spark` APIs instead |
| `spark.conf.set(...)` rejected | Some configs can't be changed on serverless. Note which ones |
| `NameError: spark is not defined` | Attach the notebook to compute and re-run |
| | |

## Interview Q&A

1. **What is SparkSession?**
   The unified entry point to Spark. It is used to create DataFrames, read data, run SQL and manage configuration.

2. **What is the difference between SparkSession and SparkContext?**
   SparkContext is the older low-level entry point that connects to the cluster and creates RDDs. SparkSession, introduced in Spark 2.0, wraps SparkContext, SQLContext and HiveContext into one object.

3. **What does `getOrCreate()` do?**
   Returns the existing SparkSession if there is one, otherwise creates a new one.

4. **Do you need to create a SparkSession in Databricks?**
   No, a session called `spark` is created for you.

5. **Can you have multiple SparkSessions?**
   Yes. They can share one SparkContext but have separate SQL configs and temp views. In practice most applications use one.

6. **Where does the SparkSession run?**
   In the driver. The executors only run tasks.

## Practice problem

Without copying from above:

1. Write the code to create a SparkSession named `PracticeApp` using the builder pattern (as a script you would run outside Databricks)
2. Use `spark` to create a DataFrame of 3 cities with columns `id` and `city`
3. Use `spark.sql` to return today's date with `SELECT current_date()`
4. Print the current catalog and database
5. In one sentence: what is the difference between SparkSession and SparkContext?

In [0]:
### Build And Practice Here 

## Summary

- SparkSession is the single entry point to Spark
- It gives you `createDataFrame`, `read`, `sql`, `conf` and `catalog`
- Databricks creates it for you as `spark`
- `getOrCreate()` reuses an existing session
- SparkSession replaced SQLContext and HiveContext, and wraps SparkContext
- On serverless, `sparkContext` access is restricted
